In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [7]:
documents = documents_llm

In [8]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
import json
user_prompt = json.dumps(doc)

In [13]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
response.output_parsed.questions

['I found this course late — can I still sign up and follow along?',
 'Is it too late to join the course if I just discovered it now?',
 'Can I still participate in the course even if I missed the start?',
 'If I join late, am I still eligible for a certificate?',
 'What do I need to do to make sure I can get the certificate if I’m joining now?']

In [17]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [18]:
from evaluation_utils import llm_structured

In [19]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join now, or is it too late?', 'If I start the course late, will I still be able to get a certificate?', 'Is it okay to join the course after it already started?', 'Do I need to submit my project before submissions close if I want the certificate?', 'Can I still take part in the course, even if I missed the beginning?']


In [20]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=302)

In [21]:
from evaluation_utils import calc_price

In [22]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00042750000000000004,
 'total_cost': 0.00058275}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still join now, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course late, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course after it already started?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take part in the course, even if I missed the beginning?',
  'document': '74eb249bbf'}]

In [24]:
import pandas as pd

In [25]:
pd.DataFrame(records)

,question,document
0,I just found this course — can I still join no...,74eb249bbf
1,"If I start the course late, will I still be ab...",74eb249bbf
2,Is it okay to join the course after it already...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,"Can I still take part in the course, even if I...",74eb249bbf


In [26]:
from evaluation_utils import llm_structured_retry

In [27]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [28]:
generate_ground_truth(doc)

([{'question': 'I just found this course — is it still okay to join now, or am I too late?',
   'document': '74eb249bbf'},
  {'question': 'Can I start the course after it already began and still participate normally?',
   'document': '74eb249bbf'},
  {'question': 'If I’m joining late, is there anything special I need to do to be eligible for the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Does the course allow new students to enter midway through, and what happens with the final project?',
   'document': '74eb249bbf'},
  {'question': 'I missed the start date — can I still take the course, and how does that affect getting a certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=319))

In [29]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [30]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [33]:
ground_truth[10]

{'question': 'Where can I watch the Office Hours or live workshop stream as a student, since I don’t get the Zoom link?',
 'document': '489dd1c9d9'}

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08776050000000003

In [35]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08776050000000003

In [36]:
df_ground_truth = pd.DataFrame(ground_truth)

In [44]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [45]:
len(df_ground_truth)

565